In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")

aws_warm = pd.read_csv(DATA_DIR / "aws_warm_aggregated.csv")
gcp_warm = pd.read_csv(DATA_DIR / "gcp_warm_aggregated.csv")
acc      = pd.read_csv(DATA_DIR / "fire_accuracy.csv")

MEMORIES = [128, 256, 512, 768, 1024, 1536, 2048]

# Basis function letter IDs (in display order for fig1)
TASKS = [
    ("a_passthrough",     "P"),
    ("b_compress",        "C"),
    ("c_decompress",      "D"),
    ("e_matrix_multiply", "M"),
    ("g_reduce",          "R"),
    ("h_expand",          "E"),
    ("k_latency_bound",   "L"),
]

# Chain mapping for accuracy figure
CHAIN_MAP = {
    'ae_passthrough_matrix_multiply': r'P{\to}M',
    'ce_decompress_matrix_multiply':  r'D{\to}M',
    'gh_reduce_expand':               r'R{\to}E',
    'kb_latency_compress':            r'L{\to}C',
}
CHAIN_ORDER = list(CHAIN_MAP.values())

def safe_col(wm, col, fallback):
    if col in wm.index and not pd.isna(wm[col]):
        return wm[col]
    return wm[fallback]

def get_breakdown(df, task, payload="medium"):
    rows = []
    for mem in MEMORIES:
        sub = df[(df['task']==task) & (df['payload_size']==payload) & (df['memory_mb']==mem)]
        if sub.empty: continue
        wm = sub.median(numeric_only=True)
        dl = safe_col(wm, 'primary_download_time_ms', 'download_time_ms')
        ul = safe_col(wm, 'primary_upload_time_ms',   'upload_time_ms')
        cp = wm['compute_time_ms']
        rows.append({'mem': mem, 'dl': dl, 'cp': cp, 'ul': ul,
                     'total_ms': wm['total_time_ms'],
                     'cost_proxy': wm['total_time_ms'] * mem})
    if not rows: return None
    df_r = pd.DataFrame(rows)
    opt  = df_r.loc[df_r['cost_proxy'].idxmin()]
    t    = opt['dl'] + opt['cp'] + opt['ul']
    return dict(dl=opt['dl']/t, cp=opt['cp']/t, ul=opt['ul']/t)

print("Setup complete.")


Setup complete.


## Fig1 — AWS Download (`\\addplot` for dl_frac)

In [3]:
lines = ["\\addplot[fill=green, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(aws_warm, task)
    lines.append(f"  ({letter}, {bd['dl']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=green, draw=none] coordinates {
  (P, 0.2337)
  (C, 0.0659)
  (D, 0.1192)
  (M, 0.1012)
  (R, 0.8805)
  (E, 0.0458)
  (L, 0.7242)
};


## Fig1 — AWS Compute (`\\addplot` for cp_frac)

In [4]:
lines = ["\\addplot[fill=colCP, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(aws_warm, task)
    lines.append(f"  ({letter}, {bd['cp']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=colCP, draw=none] coordinates {
  (P, 0.0000)
  (C, 0.8418)
  (D, 0.4620)
  (M, 0.6870)
  (R, 0.0483)
  (E, 0.2121)
  (L, 0.0641)
};


## Fig1 — AWS Upload (`\\addplot` for ul_frac)

In [5]:
lines = ["\\addplot[fill=colUL, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(aws_warm, task)
    lines.append(f"  ({letter}, {bd['ul']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=colUL, draw=none] coordinates {
  (P, 0.7663)
  (C, 0.0923)
  (D, 0.4188)
  (M, 0.2118)
  (R, 0.0711)
  (E, 0.7421)
  (L, 0.2117)
};


## Fig1 — GCP Download (`\\addplot` for dl_frac)

In [6]:
lines = ["\\addplot[fill=colDL, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(gcp_warm, task)
    lines.append(f"  ({letter}, {bd['dl']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=colDL, draw=none] coordinates {
  (P, 0.6043)
  (C, 0.1981)
  (D, 0.4551)
  (M, 0.3226)
  (R, 0.9804)
  (E, 0.0629)
  (L, 0.8145)
};


## Fig1 — GCP Compute (`\\addplot` for cp_frac)

In [7]:
lines = ["\\addplot[fill=colCP, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(gcp_warm, task)
    lines.append(f"  ({letter}, {bd['cp']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=colCP, draw=none] coordinates {
  (P, 0.0000)
  (C, 0.7005)
  (D, 0.2359)
  (M, 0.4682)
  (R, 0.0017)
  (E, 0.2316)
  (L, 0.1040)
};


## Fig1 — GCP Upload (`\\addplot` for ul_frac)

In [8]:
lines = ["\\addplot[fill=colUL, draw=none] coordinates {"]
for task, letter in TASKS:
    bd = get_breakdown(gcp_warm, task)
    lines.append(f"  ({letter}, {bd['ul']:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot[fill=colUL, draw=none] coordinates {
  (P, 0.3957)
  (C, 0.1013)
  (D, 0.3090)
  (M, 0.2091)
  (R, 0.0180)
  (E, 0.7055)
  (L, 0.0815)
};


## Accuracy — AWS micro+small (`\\addplot` mean ± std)

In [24]:
CHAINS_LATEX_EXPR = [r'P$\to$M', r'D$\to$M', r'R$\to$E', r'L$\to$C']
acc['chain'] = acc['fused_task'].map({
    'ae_passthrough_matrix_multiply': r'P$\to$M',
    'ce_decompress_matrix_multiply':  r'D$\to$M',
    'gh_reduce_expand':               r'R$\to$E',
    'kb_latency_compress':            r'L$\to$C',
})
sub = acc[(acc['provider']=='AWS') & (acc['payload_size'].isin(['micro','small']))]
lines = ["% AWS micro+small accuracy ({PAYLOAD} payload, mean +/- 1 std)",
         "\\addplot+[fill=gray!35, draw=gray!50,",
         "  error bars/.cd, y dir=both, y explicit,",
         "] coordinates {"]
for chain in CHAINS_LATEX_EXPR:
    cs = sub[sub['chain']==chain]
    m, s = cs['err_cost_pct'].mean(), cs['err_cost_pct'].std()
    lines.append(f"  ({chain}, {m:.4f}) +- (0, {s:.4f})")
lines.append("};")
print("\n".join(lines))


% AWS micro+small accuracy ({PAYLOAD} payload, mean +/- 1 std)
\addplot+[fill=gray!35, draw=gray!50,
  error bars/.cd, y dir=both, y explicit,
] coordinates {
  (P$\to$M, 0.0864) +- (0, 0.0816)
  (D$\to$M, 0.1114) +- (0, 0.1194)
  (R$\to$E, 0.1100) +- (0, 0.1340)
  (L$\to$C, 0.4514) +- (0, 0.4135)
};


## Accuracy — AWS medium+large (`\\addplot` mean ± std)

In [25]:
sub = acc[(acc['provider']=='AWS') & (acc['payload_size'].isin(['medium','large']))]
lines = ["% AWS medium+large accuracy ({PAYLOAD} payload, mean +/- 1 std)",
         "\\addplot+[fill=gray!75, draw=gray!80,",
         "  error bars/.cd, y dir=both, y explicit,",
         "] coordinates {"]
for chain in CHAINS_LATEX_EXPR:
    cs = sub[sub['chain']==chain]
    m, s = cs['err_cost_pct'].mean(), cs['err_cost_pct'].std()
    lines.append(f"  ({chain}, {m:.4f}) +- (0, {s:.4f})")
lines.append("};")
print("\n".join(lines))


% AWS medium+large accuracy ({PAYLOAD} payload, mean +/- 1 std)
\addplot+[fill=gray!75, draw=gray!80,
  error bars/.cd, y dir=both, y explicit,
] coordinates {
  (P$\to$M, 0.3271) +- (0, 0.2188)
  (D$\to$M, 0.3600) +- (0, 0.3172)
  (R$\to$E, 0.7775) +- (0, 0.9263)
  (L$\to$C, 0.9187) +- (0, 0.8403)
};


## Accuracy — GCP micro+small (`\\addplot` mean ± std)

In [21]:
sub = acc[(acc['provider']=='GCP') & (acc['payload_size'].isin(['micro','small']))]
lines = ["% GCP micro + small accuracy"
    "\\addplot+[fill=gray!35, draw=gray!50,",
         "  error bars/.cd, y dir=both, y explicit,",
         "] coordinates {"]
for chain in CHAINS_LATEX_EXPR:
    cs = sub[sub['chain']==chain]
    m, s = cs['err_cost_pct'].mean(), cs['err_cost_pct'].std()
    lines.append(f"  ({chain}, {m:.4f}) +- (0, {s:.4f})")
lines.append("};")
print("\n".join(lines))


\addplot+[fill=gray!35, draw=gray!50,
  error bars/.cd, y dir=both, y explicit,
] coordinates {
  (P$\to$M, 0.2400) +- (0, 0.2308)
  (D$\to$M, 0.4915) +- (0, 0.5500)
  (R$\to$E, 0.3267) +- (0, 0.3398)
  (L$\to$C, 1.4642) +- (0, 1.5517)
};


## Accuracy — GCP medium+large (`\\addplot` mean ± std)

In [23]:
sub = acc[(acc['provider']=='GCP') & (acc['payload_size'].isin(['medium','large']))]
lines = ["% GCP medium+large accuracy",
        "\\addplot+[fill=gray!75, draw=gray!80,",
         "  error bars/.cd, y dir=both, y explicit,",
         "] coordinates {"]
for chain in CHAINS_LATEX_EXPR:
    cs = sub[sub['chain']==chain]
    m, s = cs['err_cost_pct'].mean(), cs['err_cost_pct'].std()
    lines.append(f"  ({chain}, {m:.4f}) +- (0, {s:.4f})")
lines.append("};")
print("\n".join(lines))


% GCP medium+large accuracy
\addplot+[fill=gray!75, draw=gray!80,
  error bars/.cd, y dir=both, y explicit,
] coordinates {
  (P$\to$M, 2.0286) +- (0, 2.3627)
  (D$\to$M, 4.9186) +- (0, 3.9651)
  (R$\to$E, 1.7314) +- (0, 0.7946)
  (L$\to$C, 3.9313) +- (0, 1.4880)
};
